In [33]:
import pandas as pd


baseline_eval_path = "20251121_055142-anli_test_baseline-1/eval_predictions.jsonl"
weighted_eval_path = "20251121_055405-anli_test_weighted-1/eval_predictions.jsonl"

def compare_model_evaluations(model_a_evaluation_path, model_b_evaluation_path, model_a_name, model_b_name, model_a_suffix, model_b_suffix):
    df_a = pd.read_json(model_a_evaluation_path, lines=True)
    df_b = pd.read_json(model_b_evaluation_path, lines=True)

    # Optional: add an explicit model name column if you want
    df_baseline["model"] = model_a_name
    df_weighted["model"] = model_b_name

    merged = df_baseline.merge(
    df_weighted,
    on="uid",
    suffixes=("_"+model_a_suffix, "_"+model_b_suffix),
    how="inner",  # inner join ensures we only keep shared uids
    )

    merged["same_prediction"] = merged["predicted_label_"+model_a_suffix] == merged["predicted_label_"+model_b_suffix]

    agreement_rate = merged["same_prediction"].mean()
    print(f"Fraction of examples where models match: {agreement_rate:.3f}")

    merged["correct_"+model_a_suffix] = merged["predicted_label_"+model_a_suffix] == merged["label_"+model_a_suffix]
    merged["correct_"+model_b_suffix] = merged["predicted_label_"+model_b_suffix] == merged["label_"+model_a_suffix]

    print("Agreement per gold label:")
    print(
        merged.groupby("label_baseline")["same_prediction"].mean()
    )

    print("Break down error patterns")
    merged["both_correct"] = merged["correct_" + model_a_suffix] & merged["correct_" + model_b_suffix]
    merged["both_wrong"]  = (~merged["correct_" + model_a_suffix]) & (~merged["correct_" + model_b_suffix])
    merged[model_a_suffix + "_only_correct"] = merged["correct_" + model_a_suffix] & (~merged["correct_" + model_b_suffix])
    merged[model_b_suffix + "_only_correct"] = merged["correct_" + model_b_suffix] & (~merged["correct_" + model_b_suffix])

    summary = {
    "both_correct": merged["both_correct"].mean(),
    "both_wrong": merged["both_wrong"].mean(),
    "A_only_correct": merged[model_a_suffix + "_only_correct"].mean(),
    "B_only_correct": merged[model_b_suffix + "_only_correct"].mean(),
    }

    for k, v in summary.items():
        print(f"{k}: {v:.3f}")

    print("See where they disagree")
    disagreements = merged[merged["predicted_label_" + model_a_suffix] != merged["predicted_label_" + model_b_suffix]]

    print("Num disagreements:", len(disagreements))
    print(disagreements[["uid", "label_"+model_a_suffix, "predicted_label_" + model_a_suffix, "predicted_label_" + model_b_suffix]].head())
        
    # a_right_b_wrong = disagreements[disagreements[model_a_suffix + "_only_correct"]]
    # b_right_a_wrong = disagreements[disagreements[model_b_suffix + "_only_correct"]]

    # # Inspect a few examples
    # cols_to_show = [
    #     "uid", 
    #     "label_"+model_a_suffix,
    #     "premise_"+model_a_suffix, "hypothesis_"+model_a_suffix,  # premise/hypothesis from A (same as B if merge is clean)
    #     "predicted_label_"+model_a_suffix, "predicted_scores_"+model_a_suffix,
    #     "predicted_label_"+model_b_suffix, "predicted_scores_"+model_b_suffix,
    # ]

    # print(a_right_b_wrong[cols_to_show].head(10).to_string(index=False))
    # print(b_right_a_wrong[cols_to_show].head(10).to_string(index=False))
    

    cross = pd.crosstab(
        merged["predicted_label_"+model_a_suffix],
        merged["predicted_label_"+model_b_suffix],
        normalize="all"  # drop this if you want counts instead of fractions
    )

    print("cross")
    print(cross)

    wrong = merged[~merged["correct_"+model_a_suffix] & ~merged["correct_"+model_b_suffix]]

    wrong_cross = pd.crosstab(
        wrong["predicted_label_"+model_a_suffix],
        wrong["predicted_label_"+model_b_suffix],
        normalize="all"
    )

    print("wrong cross")
    print(wrong_cross)

compare_model_evaluations(baseline_eval_path, weighted_eval_path, "Baseline MNLI", "Weighted MNLI", 'baseline', 'weighted')

    

Fraction of examples where models match: 0.855
Agreement per gold label:
label_baseline
0    0.855148
1    0.856920
2    0.853637
Name: same_prediction, dtype: float64
Break down error patterns
both_correct: 0.590
both_wrong: 0.291
A_only_correct: 0.065
B_only_correct: 0.000
See where they disagree
Num disagreements: 2449
                                     uid  label_baseline  \
24  75f42744-a530-4948-8041-5126da7f7400               2   
31  e1fea81e-c1f7-4d4f-bc4b-d0100932b4ee               1   
32  7d64c39c-fb18-4fbf-b2a7-1b627a6409c2               1   
41  9e06d36b-e490-4b28-b8b0-037fd4bc05d9               0   
49  b18e5822-f45f-44f3-99ab-3b4d8a915b52               1   

    predicted_label_baseline  predicted_label_weighted  
24                         2                         0  
31                         1                         0  
32                         1                         2  
41                         1                         0  
49                         1  